# RAON Full-Duplex Inference

This notebook demonstrates full-duplex (simultaneous listen-and-speak) inference using the RAON duplex model.

**Features:**
- **Speak-first / Listen-first**: Runtime token forcing controls whether the model speaks first or waits
- **Persona catalog**: 17 preset personas (general, scenario_restaurant, scenario_movie, etc.)
- **Custom system prompt**: Arbitrary prompt text

**Outputs per run:**
- `assistant.wav` — model-generated speech
- `user_assistant.wav` — stereo mix (L=user, R=assistant)
- `output.json` — duration/sample count summary
- `frame_log.txt` — per-frame text/audio debug log

In [ ]:
# Requirements:
#   pip install transformers>=4.57.1 torch torchaudio soundfile accelerate
#
# For local model loading, also install raon:
#   pip install -e .   (or: uv sync)
#
# Optional: install FlashAttention 2 for lower memory usage
#   pip install -U flash-attn --no-build-isolation

## Model Loading

Choose **one** of the two options below:
- **Option A**: Load from HuggingFace Hub (no `pip install raon` needed)
- **Option B**: Load from a local checkpoint path (requires `pip install -e .`)

### Option A: Load from HuggingFace Hub

No `pip install raon` needed — only `transformers`, `torch`, `torchaudio`, `soundfile`.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import torch
from IPython.display import Audio, display
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_ID = "KRAFTON/Raon-SpeechChat-9B"
DEVICE = "cuda"
DTYPE = "bfloat16"
AUDIO_INPUT = "../data/duplex/eval/audio/duplex_00.wav"
AUDIO_INPUT_PERSONA = "../data/duplex/eval/audio/duplex_01.wav"
SPEAKER_REF_AUDIO = "../data/duplex/eval/audio/spk_ref.wav"
OUTPUT_ROOT = Path("output/duplex_notebook")

# Resolve Hub module
print(f"Resolving Hub module from {MODEL_ID}...")
_cfg = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
_revision = getattr(_cfg, "_commit_hash", None)
RaonPipeline = get_class_from_dynamic_module(
    "modeling_raon.RaonPipeline",
    MODEL_ID,
    revision=_revision,
)
build_system_prompt = get_class_from_dynamic_module(
    "modeling_raon.build_system_prompt",
    MODEL_ID,
    revision=_revision,
)
del _cfg, _revision

# Create pipeline
print("Creating RaonPipeline...")
pipe = RaonPipeline(MODEL_ID, device=DEVICE, dtype=DTYPE)
print("Pipeline ready.")

### Option B: Load from Local Path

Requires `pip install -e .` (or `uv sync`) from the repository root.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

from IPython.display import Audio, display

from raon import RaonPipeline
from raon.utils.duplex_prompt_catalog import build_system_prompt

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_PATH = "/path/to/duplex-model"  # <-- set this
DEVICE = "cuda"
DTYPE = "bfloat16"
AUDIO_INPUT = "../data/duplex/eval/audio/duplex_00.wav"
AUDIO_INPUT_PERSONA = "../data/duplex/eval/audio/duplex_01.wav"
SPEAKER_REF_AUDIO = "../data/duplex/eval/audio/spk_ref.wav"
OUTPUT_ROOT = Path("output/duplex_notebook")

# Load the pipeline
pipe = RaonPipeline(MODEL_PATH, device=DEVICE, dtype=DTYPE)
print("Pipeline ready.")

---

## Load Audio Input

This notebook uses duplex eval mono audio (`../data/duplex/eval/audio/*.wav`).
For inference, pass the waveform as-is (no channel selection).

In [ ]:
user_audio = pipe.load_audio(AUDIO_INPUT)
display(Audio(user_audio[0].float().cpu().numpy(), rate=pipe.processor.sampling_rate))

## Helper: Display Results

In [ ]:
def run_and_display(label: str, audio_input_path: str | None = None, **kwargs) -> dict:
    """Run duplex inference via pipeline and display results inline."""
    out_dir = OUTPUT_ROOT / label.lower().replace(" ", "_")
    kwargs.setdefault("speaker_audio", SPEAKER_REF_AUDIO)

    print(f"\n{'=' * 60}")
    print(f"  {label}")
    for k, v in kwargs.items():
        print(f"  {k}: {v!r}")
    print(f"{'=' * 60}")

    audio = pipe.load_audio(audio_input_path) if audio_input_path else user_audio
    summary = pipe.duplex(audio_input=audio, output_dir=str(out_dir), **kwargs)

    frame_log = (out_dir / "frame_log.txt").read_text()
    tokens = re.findall(r"text='([^']+)'", frame_log)
    decoded = "".join(t for t in tokens if t != "-")

    print(f"\nDecoded text: {decoded}")
    print(f"Duration: {summary['assistant_duration_sec']:.1f}s")
    display(Audio(str(out_dir / "assistant.wav")))

    return summary

## Listen-First

The model does not begin speaking immediately. First `[U]` prediction is forced to SIL (silence).

In [ ]:
_ = run_and_display("Listen First", audio_input_path=AUDIO_INPUT)

## Speak-First

The model begins speaking immediately. First `[U]` prediction is forced to EPAD (speech onset).

In [ ]:
_ = run_and_display("Speak First", audio_input_path=AUDIO_INPUT, speak_first=True)

## Persona: Restaurant Assistant

Using the persona catalog to set a restaurant assistant character.

In [ ]:
_ = run_and_display(
    "Persona Restaurant",
    audio_input_path=AUDIO_INPUT_PERSONA,
    system_prompt=build_system_prompt(persona="scenario_restaurant"),
)

## Persona: General Assistant

In [ ]:
_ = run_and_display(
    "Persona General",
    audio_input_path=AUDIO_INPUT_PERSONA,
    system_prompt=build_system_prompt(persona="general"),
)

## Persona with Context

Adding extra context to the persona prompt.

In [ ]:
prompt = build_system_prompt(
    persona="scenario_restaurant",
    context="Discuss restaurant menu choices and delivery preferences with a customer.",
)
print(f"System prompt: {prompt!r}\n")

_ = run_and_display("Persona Context", audio_input_path=AUDIO_INPUT_PERSONA, system_prompt=prompt)

## Custom System Prompt

Bypass the persona catalog and provide any prompt text directly.

In [ ]:
_ = run_and_display(
    "Custom Prompt",
    audio_input_path=AUDIO_INPUT_PERSONA,
    system_prompt="You are Raon, a friendly restaurant assistant. Help the user choose a meal, check delivery timing, and confirm dietary preferences.",
)

## Cleanup

In [ ]:
del pipe, user_audio
import gc
gc.collect()
torch.cuda.empty_cache()
print("Cleaned up.")